# Level 3 — Macro Regime + Earnings Roadmap

## ปัญหาของ Level 1-2
```
Level 2 Walk-Forward 2022-04→2022-10: Sharpe = -1.26  ← worst period
2022 = SPY ลง 20%, SPY > SMA200 แค่ 18.7% ของวัน
→ เราเป็น LONG-only model ที่ trade ใน bear market = แพ้แน่นอน
```

## Level 3 Upgrades

| Option | Alpha Source | ผลที่คาด | ความยาก |
|---|---|---|---|
| **2 (ทำก่อน)** | SPY + VIX macro regime | Sharpe +0.2–0.4 | ง่าย (30 นาที) |
| **1 (ต่อไป)** | Earnings Surprise (PEAD) | Sharpe +0.5–1.0 | กลาง (1-2 วัน) |
| 3 | Intraday hourly data | สูง แต่ infrastructure ใหม่ | ยาก (หลายสัปดาห์) |

---

## OPTION 2: Cross-Asset Macro Regime

```
SPY > SMA200  → Bull market → trade normally
SPY < SMA200  → Bear market → ไม่ trade (หรือลด size มาก)

VIX < 20      → Risk-on  → full position
VIX 20-30     → Cautious → 70% position  
VIX > 30      → Risk-off → 30% position
VIX > 40      → Crisis   → 10% position
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# ── Load stock features ───────────────────────────────────────────────────
df = pd.read_csv('../../data/processed/features/features_all.csv', parse_dates=['Date'])
df = df.sort_values(['symbol', 'Date']).reset_index(drop=True)
FEATURES = [c for c in df.columns if c not in
            ['Date', 'symbol', 'target', 'Open', 'High', 'Low', 'Close', 'Volume']]

train_base = df[df['Date'] < '2022-01-01']
mf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
mf.fit(train_base[FEATURES], train_base['target'])
imp = pd.Series(mf.feature_importances_, index=FEATURES).sort_values(ascending=False)
TOP = imp.head(15).index.tolist()

# ── Download SPY + VIX ───────────────────────────────────────────────────
print('Downloading SPY and VIX...')
spy_raw = yf.download('SPY', start='2015-01-01', end='2025-01-01',
                      auto_adjust=True, progress=False)['Close'].squeeze()
vix_raw = yf.download('^VIX', start='2015-01-01', end='2025-01-01',
                      auto_adjust=True, progress=False)['Close'].squeeze()

print(f'SPY: {len(spy_raw)} days  |  VIX: {len(vix_raw)} days')
print(f'Bull market (SPY>SMA200): {float((spy_raw > spy_raw.rolling(200).mean()).mean()):.1%} of all days')
print(f'2022 bull days: {float((spy_raw["2022"] > spy_raw["2022"].rolling(200).mean()).mean()):.1%}  ← bear year')

---
## Section 1 — Build Macro Regime Features (lag-1)

ทุก feature ใช้ **lag-1** (yesterday's value) เพื่อไม่มี look-ahead bias

In [ ]:
def build_macro_regime(spy: pd.Series, vix: pd.Series) -> pd.DataFrame:
    """
    Build daily macro regime features (all lag-1).
    Returns DataFrame indexed by Date.
    """
    macro = pd.DataFrame(index=spy.index)

    # SPY trend regime
    spy_sma50  = spy.rolling(50).mean()
    spy_sma200 = spy.rolling(200).mean()
    macro['spy_bull']       = (spy > spy_sma200).astype(float)   # above 200-MA
    macro['spy_momentum']   = spy.pct_change(20)                  # 20-day return
    macro['spy_above_50']   = (spy > spy_sma50).astype(float)

    # VIX regime
    macro['vix']            = vix
    macro['vix_low']        = (vix < 20).astype(float)           # risk-on
    macro['vix_high']       = (vix > 30).astype(float)           # risk-off

    # VIX position size multiplier (continuous)
    macro['vix_size_mult']  = np.where(
        vix < 20,  1.0,
        np.where(vix < 30, 0.7,
        np.where(vix < 40, 0.3, 0.1))
    )

    # All shifted by 1 day (lag-1 — no look-ahead)
    macro = macro.shift(1)
    macro.index.name = 'Date'   # normalise index name (yfinance may return 'Datetime')
    return macro


macro_df = build_macro_regime(spy_raw, vix_raw)

# Merge macro into stock dataframe
df_m = df.merge(macro_df.reset_index(), on='Date', how='left')

# Handle missing macro (weekends fill from Friday)
macro_cols = ['spy_bull', 'spy_momentum', 'spy_above_50',
              'vix', 'vix_low', 'vix_high', 'vix_size_mult']
df_m[macro_cols] = df_m[macro_cols].ffill()

print(f'df_m shape: {df_m.shape}')
print(f'Macro coverage: {df_m["spy_bull"].notna().mean():.1%}')
print()

# Win rate by macro regime — key validation
test_m = df_m[df_m['Date'] >= '2018-01-01']
print('=== WIN RATE BY MACRO REGIME ===')
for bull in [0, 1]:
    for vix_h in [0, 1]:
        mask = (test_m['spy_bull'] == bull) & (test_m['vix_high'] == vix_h)
        n    = mask.sum()
        if n < 100: continue
        wr   = test_m.loc[mask, 'target'].mean()
        label = f'SPY_bull={bull}  VIX_high={vix_h}'
        print(f'  {label}  n={n:5,}  target_win_rate={wr:.3f}')

print()
print('  → SPY_bull=1 (uptrend) should have higher win rate')
print('  → VIX_high=0 (low fear) should have higher win rate')

---
## Section 2 — Macro-Enhanced Simulator

In [ ]:
class MacroSimulator:
    """
    Level 3: combines Level 2 (Kelly + per-stock regime)
    with macro regime filter (SPY + VIX).

    Entry rules:
      1. proba - 0.5 >= min_edge
      2. stock trend_ok (Close > SMA50, per-stock)
      3. spy_bull == 1  (SPY above 200-MA)       ← NEW
    Position size:
      Kelly × vix_size_mult                       ← NEW
    """

    def __init__(self, initial_capital=100_000, max_positions=5,
                 kelly_fraction=0.25, max_position_pct=0.15,
                 hold_days=5, tx_cost=0.0005, stop_loss=-0.07,
                 min_edge=0.02):
        self.initial_capital  = initial_capital
        self.max_positions    = max_positions
        self.kelly_fraction   = kelly_fraction
        self.max_position_pct = max_position_pct
        self.hold_days        = hold_days
        self.tx_cost          = tx_cost
        self.stop_loss        = stop_loss
        self.min_edge         = min_edge

    def kelly_pos(self, edge, wr, aw, al):
        if al >= 0: return 0.0
        b = aw / abs(al)
        full_k = max(0.0, (wr * b - (1 - wr)) / b)
        return min(full_k * self.kelly_fraction * min(edge / 0.10, 1.5),
                   self.max_position_pct)

    def simulate(self, df, probas, wr=0.54, aw=0.025, al=-0.020):
        df = df.copy().reset_index(drop=True)
        df['proba']  = np.asarray(probas)
        df['edge']   = df['proba'] - 0.5

        # Ensure per-stock regime exists
        if 'trend_ok' not in df.columns:
            df = _add_stock_regime(df)

        # Signal: edge + per-stock trend + SPY bull
        df['signal'] = (
            (df['edge'] >= self.min_edge) &
            (df['trend_ok'].fillna(0) == 1) &
            (df['spy_bull'].fillna(0) == 1)      # macro gate
        ).astype(int)

        price_map = df.set_index(['Date', 'symbol'])['Close'].to_dict()
        all_dates = sorted(df['Date'].unique())
        date_to_i = {d: i for i, d in enumerate(all_dates)}

        capital = float(self.initial_capital)
        positions = []
        daily_rows, trade_rows = [], []

        for today in all_dates:
            today_i    = date_to_i[today]
            today_data = df[df['Date'] == today]

            surviving = []
            for pos in positions:
                dh = today_i - pos['entry_i']
                ep = price_map.get((today, pos['symbol']), pos['entry_price'])
                gr = ep / pos['entry_price'] - 1
                if dh >= self.hold_days or gr <= self.stop_loss:
                    net = gr - self.tx_cost
                    capital += pos['value'] * (1 + net)
                    trade_rows.append({
                        'symbol': pos['symbol'], 'entry_date': pos['entry_date'],
                        'exit_date': today, 'net_return': net,
                        'edge': pos['edge'],
                        'exit_reason': 'stop' if gr <= self.stop_loss else 'time',
                    })
                else:
                    surviving.append(pos)
            positions = surviving

            capacity  = self.max_positions - len(positions)
            open_syms = {p['symbol'] for p in positions}

            if capacity > 0:
                for _, row in today_data[today_data['signal'] == 1].sort_values('edge', ascending=False).iterrows():
                    if capacity <= 0: break
                    if row['symbol'] in open_syms: continue

                    pos_pct = self.kelly_pos(row['edge'], wr, aw, al)
                    # Scale by VIX multiplier
                    vix_mult = row.get('vix_size_mult', 1.0)
                    if pd.isna(vix_mult): vix_mult = 1.0
                    pos_pct *= vix_mult

                    tv = capital * pos_pct
                    if tv < 100: continue

                    capital -= tv * (1 + self.tx_cost)
                    positions.append({
                        'symbol': row['symbol'], 'entry_date': today,
                        'entry_i': today_i, 'entry_price': row['Close'],
                        'value': tv, 'edge': row['edge'],
                    })
                    open_syms.add(row['symbol'])
                    capacity -= 1

            mtm = sum(p['value'] * price_map.get((today, p['symbol']), p['entry_price'])
                      / p['entry_price'] for p in positions)
            daily_rows.append({'date': today, 'portfolio_value': capital + mtm,
                               'n_positions': len(positions),
                               'spy_bull': today_data['spy_bull'].iloc[0] if len(today_data) > 0 else 0})

        daily_df  = pd.DataFrame(daily_rows).set_index('date')
        trades_df = pd.DataFrame(trade_rows)
        return daily_df, trades_df

    @staticmethod
    def metrics(daily_df, trades_df, rf=0.04):
        pv   = daily_df['portfolio_value']
        ret  = pv.pct_change().fillna(0)
        peak = pv.cummax(); dd = (pv - peak) / peak
        exc  = ret - rf / 252
        sh   = exc.mean() / exc.std() * np.sqrt(252) if exc.std() > 0 else 0
        tr   = pv.iloc[-1] / pv.iloc[0] - 1
        ny   = (pv.index[-1] - pv.index[0]).days / 365.25
        cagr = (1 + tr) ** (1 / ny) - 1 if ny > 0 else 0
        r    = trades_df['net_return'] if len(trades_df) else pd.Series(dtype=float)
        w    = r[r > 0]; l = r[r < 0]
        pf   = w.sum() / abs(l.sum()) if len(l) > 0 and l.sum() != 0 else np.inf
        return {
            'n_trades': len(trades_df), 'win_rate': float((r > 0).mean()),
            'profit_factor': float(pf), 'sharpe': float(sh),
            'max_drawdown': float(dd.min()), 'cagr': float(cagr),
            '_equity': pv, '_dd': dd,
        }


def _add_stock_regime(df, tw=50, vw=20, vmw=50):
    df = df.copy(); tl, vl = [], []
    for sym, grp in df.groupby('symbol'):
        grp   = grp.sort_values('Date').reset_index(drop=True)
        close = grp['Close']
        tl.append((close > close.rolling(tw).mean()).astype(int).shift(1).reindex(grp.index))
        v20 = close.pct_change().rolling(vw).std()
        vl.append((v20 > v20.rolling(vmw).mean()).astype(int).shift(1).reindex(grp.index))
    df['trend_ok'] = pd.concat(tl).values
    df['high_vol'] = pd.concat(vl).values
    return df


print('MacroSimulator ready')

---
## Section 3 — Walk-Forward with Macro Regime

In [ ]:
INITIAL_YEARS = 3
STEP_MONTHS   = 6

df_m2 = _add_stock_regime(df_m)  # add per-stock regime to macro-merged df

min_date = df_m2['Date'].min()
max_date = df_m2['Date'].max()
cutoff   = min_date + pd.DateOffset(years=INITIAL_YEARS)

wf_periods   = []
all_daily    = []
all_trades   = []
prev_capital = 100_000

while True:
    t_start = cutoff
    t_end   = cutoff + pd.DateOffset(months=STEP_MONTHS)
    if t_end > max_date:
        break

    train_wf = df_m2[df_m2['Date'] <  t_start]
    test_wf  = df_m2[(df_m2['Date'] >= t_start) & (df_m2['Date'] < t_end)].reset_index(drop=True)

    if len(train_wf) < 500 or len(test_wf) < 50:
        cutoff = t_end
        continue

    model_wf = RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=30,
        max_features='sqrt', random_state=42, n_jobs=-1,
    )
    model_wf.fit(train_wf[TOP], train_wf['target'])

    # Estimate Kelly stats from training validation
    n_tr  = len(train_wf)
    val   = train_wf.iloc[int(n_tr * 0.8):].reset_index(drop=True)
    pb_v  = model_wf.predict_proba(val[TOP])[:, 1]
    sig_v = (pb_v - 0.5) > 0.02
    t5s   = []
    for sym, grp in val.groupby('symbol'):
        grp = grp.sort_values('Date')
        idx = grp.index.to_numpy()   # positions in val BEFORE reset — must capture here
        grp = grp.reset_index(drop=True)
        c   = grp['Close'].values
        s   = sig_v[idx]             # use val positions, not post-reset 0,1,2,...
        for i in range(len(grp) - 5):
            if s[i]: t5s.append(c[i+5]/c[i]-1-0.001)
    if len(t5s) < 10: t5s = [0.025, -0.020]
    t5 = np.array(t5s)
    wr  = float((t5 > 0).mean())
    aw  = float(t5[t5 > 0].mean()) if (t5 > 0).any() else 0.025
    al  = float(t5[t5 < 0].mean()) if (t5 < 0).any() else -0.020

    probas_wf = model_wf.predict_proba(test_wf[TOP])[:, 1]
    sim = MacroSimulator(
        initial_capital  = prev_capital,
        max_positions    = 5,
        kelly_fraction   = 0.25,
        max_position_pct = 0.15,
        hold_days        = 5,
        tx_cost          = 0.0005,
        stop_loss        = -0.07,
        min_edge         = 0.02,
    )
    daily_wf, trades_wf = sim.simulate(test_wf, probas_wf, wr, aw, al)

    if daily_wf.empty or trades_wf.empty:
        cutoff = t_end
        continue

    prev_capital = daily_wf['portfolio_value'].iloc[-1]
    mx = MacroSimulator.metrics(daily_wf, trades_wf)

    bull_pct = float(daily_wf['spy_bull'].mean()) if 'spy_bull' in daily_wf.columns else 0
    wf_periods.append({
        'period'        : t_start.strftime('%Y-%m'),
        'bull_pct'      : bull_pct,
        'n_trades'      : mx['n_trades'],
        'win_rate'      : mx['win_rate'],
        'profit_factor' : mx['profit_factor'],
        'sharpe'        : mx['sharpe'],
        'max_drawdown'  : mx['max_drawdown'],
        'cagr'          : mx['cagr'],
    })
    all_daily.append(daily_wf)
    all_trades.append(trades_wf)

    print(f'  [{t_start.strftime("%Y-%m")}→{t_end.strftime("%Y-%m")}]  '
          f'bull={bull_pct:.0%}  trades={mx["n_trades"]:3d}  '
          f'sharpe={mx["sharpe"]:+.2f}  dd={mx["max_drawdown"]:.1%}  '
          f'pf={mx["profit_factor"]:.2f}')

    cutoff = t_end

wf_df      = pd.DataFrame(wf_periods)
trades_all = pd.concat(all_trades).reset_index(drop=True) if all_trades else pd.DataFrame()
print(f'\nTotal periods: {len(wf_df)}')

In [ ]:
# Combined metrics
combined_daily  = pd.concat(all_daily)
combined_equity = combined_daily['portfolio_value']
mx_all = MacroSimulator.metrics(combined_daily, trades_all)

print(f"\n{'=' * 60}")
print(f"  LEVEL 3 (Macro Regime) — COMBINED RESULTS")
print(f"{'=' * 60}")
print(f"  Trades        : {mx_all['n_trades']:>8,}")
print(f"  Win Rate      : {mx_all['win_rate']:>8.1%}")
print(f"  Profit Factor : {mx_all['profit_factor']:>8.3f}")
print(f"  Sharpe Ratio  : {mx_all['sharpe']:>8.3f}")
print(f"  Max Drawdown  : {mx_all['max_drawdown']:>8.1%}")
print(f"  CAGR          : {mx_all['cagr']:>8.1%}")
print(f"{'=' * 60}")

print(f'\nSTABILITY')
print(f'  Periods Sharpe > 0.0 : {(wf_df["sharpe"] > 0.0).sum()}/{len(wf_df)}')
print(f'  Periods Sharpe > 0.5 : {(wf_df["sharpe"] > 0.5).sum()}/{len(wf_df)}')
print(f'  Mean Sharpe : {wf_df["sharpe"].mean():.3f}')
print(f'  Std  Sharpe : {wf_df["sharpe"].std():.3f}')

In [ ]:
# Level 1 → 2 → 3 comparison
print(f"\n{'=' * 65}")
print(f"  PROGRESSION: Level 1 → Level 2 → Level 3")
print(f"{'=' * 65}")
print(f"  {'Metric':<20} {'Level 1':>14} {'Level 2':>14} {'Level 3':>14}")
print('-' * 65)

rows = [
    ('Position size',  'Fixed 10%',      'Kelly (0.25x)',   'Kelly × VIX'),
    ('Signal filter',  'proba≥0.52',     'edge≥opt+regime', 'edge+regime+SPY'),
    ('Macro filter',   'None',           'None',            'SPY>SMA200'),
    ('VIX scaling',    'None',           'None',            'vix_size_mult'),
    ('--- RESULTS ---', '---',           '---',             '---'),
    ('Trades',         '617',            '~400',            f'{mx_all["n_trades"]}'),
    ('Win Rate',       '54.0%',          '~56%',            f'{mx_all["win_rate"]:.1%}'),
    ('Profit Factor',  '1.249',          '~1.3',            f'{mx_all["profit_factor"]:.3f}'),
    ('Sharpe',         '0.199',          '~0.3',            f'{mx_all["sharpe"]:.3f}'),
    ('Max Drawdown',   '-12.1%',         '~-10%',           f'{mx_all["max_drawdown"]:.1%}'),
    ('CAGR',           '5.6%',           '~6%',             f'{mx_all["cagr"]:.1%}'),
]
for label, lv1, lv2, lv3 in rows:
    print(f"  {label:<20} {lv1:>14} {lv2:>14} {lv3:>14}")
print('=' * 65)

In [ ]:
# Dashboard
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

# 1. Equity + SPY regime shading
ax1 = fig.add_subplot(gs[0, :])
eq_norm = combined_equity / combined_equity.iloc[0]
ax1.plot(eq_norm.index, eq_norm.values, color='steelblue', lw=2, label='Level 3 Portfolio')

bah_ret = df_m2[df_m2['Date'] >= combined_equity.index[0]].groupby('Date')['Close'].mean().pct_change().fillna(0)
bah_eq  = (1 + bah_ret.reindex(eq_norm.index, fill_value=0)).cumprod()
ax1.plot(bah_eq.index, bah_eq.values, color='gray', lw=1.2, ls='--', label='Buy-and-Hold')

# Shade bear market periods
bull_mask = combined_daily['spy_bull'].fillna(1) == 0
if bull_mask.any():
    ax1.fill_between(eq_norm.index, 0, eq_norm.max() * 1.1,
                     where=bull_mask.reindex(eq_norm.index, fill_value=False),
                     alpha=0.15, color='red', label='Bear market (SPY<SMA200)')

ax1.axhline(1, color='black', lw=0.8, ls=':')
ax1.set_title(f'Level 3 Equity  |  Sharpe={mx_all["sharpe"]:.2f}  '
              f'MaxDD={mx_all["max_drawdown"]:.1%}  CAGR={mx_all["cagr"]:.1%}\n'
              f'Red = Bear market (no trading allowed)', fontsize=12)
ax1.legend(); ax1.set_ylabel('Normalized Value')

# 2. Drawdown
ax2 = fig.add_subplot(gs[1, 0])
dd = mx_all['_dd']
ax2.fill_between(dd.index, dd.values * 100, 0, color='tomato', alpha=0.7)
ax2.axhline(-15, color='orange', lw=1, ls='--', label='-15% target')
ax2.set_title(f'Drawdown  |  Max = {dd.min():.1%}', fontsize=12)
ax2.set_ylabel('Drawdown (%)'); ax2.legend()

# 3. Sharpe per period with bull% overlay
ax3 = fig.add_subplot(gs[1, 1])
colors = ['steelblue' if s > 0 else 'tomato' for s in wf_df['sharpe']]
ax3.bar(range(len(wf_df)), wf_df['sharpe'], color=colors, edgecolor='white')
ax3_twin = ax3.twinx()
ax3_twin.plot(range(len(wf_df)), wf_df['bull_pct'] * 100, 'k--', lw=1.5,
              marker='o', ms=4, label='% Bull days')
ax3_twin.set_ylabel('Bull days (%)', color='black')
ax3.axhline(0, color='black', lw=0.8, ls=':')
ax3.set_xticks(range(len(wf_df)))
ax3.set_xticklabels(wf_df['period'], rotation=45, ha='right', fontsize=7)
ax3.set_title('Sharpe per Period (bars) vs % Bull Market Days (line)', fontsize=11)
ax3.set_ylabel('Sharpe')

# 4. VIX over time
ax4 = fig.add_subplot(gs[2, 0])
vix_aligned = vix_raw.reindex(combined_equity.index, method='ffill')
ax4.fill_between(vix_aligned.index, vix_aligned.values, alpha=0.5, color='orange')
ax4.axhline(20, color='green',  lw=1, ls='--', label='VIX=20 (risk-on threshold)')
ax4.axhline(30, color='red',    lw=1, ls='--', label='VIX=30 (risk-off threshold)')
ax4.set_title('VIX (Fear Index) — drives position size multiplier', fontsize=11)
ax4.set_ylabel('VIX'); ax4.legend(fontsize=7)

# 5. Win rate vs SPY regime
ax5 = fig.add_subplot(gs[2, 1])
if len(trades_all) > 0 and 'entry_date' in trades_all.columns:
    trades_all['entry_dt'] = pd.to_datetime(trades_all['entry_date'])
    # map entry date → spy_bull value; shift(-1) undoes the lag-1 applied in build_macro_regime
    bull_map = macro_df['spy_bull'].shift(-1).to_dict()
    trades_all['bull_at_entry'] = trades_all['entry_dt'].map(bull_map).fillna(0)
    for bull_val, label in [(1, 'Bull (SPY>SMA200)'), (0, 'Bear (SPY<SMA200)')]:
        r = trades_all[trades_all['bull_at_entry'] == bull_val]['net_return']
        if len(r) > 0:
            ax5.hist(r * 100, bins=30, alpha=0.6, label=f'{label} (n={len(r)}, wr={(r>0).mean():.0%})')
    ax5.axvline(0, color='red', lw=1.5, ls='--')
    ax5.set_xlabel('Net Return (%)')
    ax5.set_title('Trade Returns: Bull vs Bear Entry Regime', fontsize=11)
    ax5.legend(fontsize=8)

plt.suptitle('Level 3 Dashboard — Macro Regime Filter (SPY + VIX)', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('backtest_level3_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: backtest_level3_dashboard.png')

---
## OPTION 1 — Earnings Surprise (PEAD) — Design & Roadmap

### ทำไม Earnings Surprise ถึงสำคัญ?

**Post-Earnings Announcement Drift (PEAD)** เป็น anomaly ที่ documented มา 50+ ปี:
```
บริษัทประกาศกำไรเกินคาด (positive surprise)
  → ราคาขึ้นทันที (earnings day)
  → ยังคง drift ขึ้นอีก 5-20 วัน (PEAD)

เหตุผล: นักลงทุนรายย่อย under-react ต่อข่าวดี
        institutional investors ค่อยๆ ซื้อตามวันต่อๆ มา
```

### Data ที่ต้องการ

```python
# yfinance มี earnings history ฟรี
import yfinance as yf
ticker = yf.Ticker('AAPL')
earnings = ticker.earnings_history  # quarterly EPS actual vs estimate

# คำนวณ Earnings Surprise
surprise_pct = (actual_eps - estimated_eps) / abs(estimated_eps)
# Positive surprise > 5% → strong BUY signal
```

### Signal Design

```python
# สร้าง event-based signal
def earnings_signal(df_prices, earnings_df):
    """
    สร้าง signal = 1 เมื่อ:
    1. วันนี้คือ earnings date
    2. surprise_pct > threshold (e.g. 5%)
    3. SPY อยู่ใน bull regime
    
    Hold 10 วัน (PEAD มักกินเวลา 1-2 สัปดาห์)
    """
    ...
```

### Expected Results

| Metric | Technical-only | + Earnings Surprise |
|---|---|---|
| Win Rate | 54-58% | **62-70%** |
| Sharpe | 0.2-0.5 | **0.8-1.5** |
| Profit Factor | 1.2-1.4 | **1.5-2.5** |
| Trade frequency | สูง (~600/year) | ต่ำ (~50/year) แต่ quality สูง |

### Implementation Plan (1-2 วัน)

```
Step 1: Download earnings history สำหรับ 7 symbols
  → yf.Ticker('AAPL').earnings_history
  → Clean และ compute surprise_pct

Step 2: สร้าง feature matrix รวม earnings signal
  → earnings_surprise (0 ถ้าไม่มี earnings วันนั้น)
  → days_since_earnings (0-30)
  → beat_pct (% that beat in last 4 quarters)

Step 3: สร้าง pure event-driven strategy
  → ซื้อ day+1 หลัง positive surprise > 5%
  → hold 10 วัน
  → combine กับ macro regime filter

Step 4: Backtest + compare vs technical-only
```

In [ ]:
# Preview earnings data availability
print('=== EARNINGS DATA PREVIEW ===')
print('Checking yfinance earnings history for our symbols...\n')

for sym in ['AAPL', 'MSFT', 'NVDA', 'TSLA']:
    try:
        ticker = yf.Ticker(sym)
        eh = ticker.earnings_history
        if eh is not None and len(eh) > 0:
            eh = eh.dropna(subset=['epsActual', 'epsEstimate'])
            eh['surprise_pct'] = (eh['epsActual'] - eh['epsEstimate']) / eh['epsEstimate'].abs()
            pos_surprise = (eh['surprise_pct'] > 0.05).mean()
            print(f'  {sym}: {len(eh)} quarters  |  '
                  f'positive surprise (>5%): {pos_surprise:.0%}  |  '
                  f'mean surprise: {float(eh["surprise_pct"].mean()):.1%}')
        else:
            print(f'  {sym}: No earnings history available')
    except Exception as e:
        print(f'  {sym}: Error - {e}')

print()
print('→ ข้อมูล earnings พร้อมสร้าง PEAD signal')
print('→ Next step: notebook/feature/03_earnings_signal.ipynb')